In [ ]:
import os, sys
sys.path.append(os.path.join(os.environ.get('IDEFIX_DIR','../../../../'), 'pytools'))
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import dump_io as io

from astropy import constants as const, units as u

%matplotlib inline


In [ ]:
dirc = 'out/'

def load(dirc=dirc, i=0, var='Vc-RHO', slicer=np.s_[:]): return io.readDump(f'{dirc}/dump.{i:04d}.dmp').data[var].transpose(2,1,0)[slicer][0]
def load3D(dirc=dirc, i=0, var='Vc-RHO', slicer=np.s_[:]): return io.readDump(f'{dirc}/dump.{i:04d}.dmp').data[var].transpose(2,1,0)[slicer]

In [ ]:
dmp = io.readDump(f'{dirc}/dump.0000.dmp')
r, th, phi = dmp.x1, dmp.x2, dmp.x3


uL = const.au.cgs
uM = const.M_sun.cgs
uv = np.sqrt(const.G.cgs * uM / uL)
uT = ((u.g.cgs/u.mol.cgs) * uv**2 / (const.k_B.cgs * const.N_A.cgs)).decompose()
ud = uM/uL**3

mu = 2.353


In [ ]:

snap = 10

rho = load(i=snap, var='Vc-RHO')
P = load(i=snap, var='Vc-PRS')
Er = load(i=snap, var='ERAD')

T = mu * (P/rho)

z = r*np.cos(th[:,None])

fig, ax = plt.subplots(ncols=1, dpi=150, figsize=(8,3))
im = ax.pcolormesh(r, z, rho * ud.to_value('g/cm**3'), norm=LogNorm(vmin=1e-10*rho.max() *ud.to_value('g/cm**3')))
ax.set_xlabel('Radius [AU]')
ax.set_ylabel('Z/R')
fig.colorbar(im, ax=ax, label='Density [cgs]')

fig, ax = plt.subplots(ncols=3, dpi=150, figsize=(8,3))

im = ax[0].pcolormesh(r, z, T * uT.to_value('K'), norm=LogNorm())
ax[0].set_xlabel('Radius [AU]')
ax[0].set_ylabel('Z/R')
fig.colorbar(im, ax=ax[0], label='Gas Temperature [K]')

Td = load(i=snap, var='Dust0_Vc-TR0')
im = ax[1].pcolormesh(r, z, Td * uT.to_value('K'), norm=LogNorm())  
ax[1].set_xlabel('Radius [AU]')
ax[1].set_ylabel('Z/R')
fig.colorbar(im, ax=ax[1], label='Large Dust Temperature [K]')

Td = load(i=snap, var='Dust1_Vc-TR0')
im = ax[2].pcolormesh(r, z, Td * uT.to_value('K'), norm=LogNorm())  
ax[2].set_xlabel('Radius [AU]')
ax[2].set_ylabel('Z/R')
fig.colorbar(im, ax=ax[2], label='Small Dust Temperature [K]')

fig.tight_layout()


In [ ]:
fig, ax = plt.subplots(ncols=1, dpi=150)

T = mu * (load3D(i=snap, var='Vc-PRS')/load3D(i=snap, var='Vc-RHO'))

for i in range(T.shape[0]):
    ax.loglog(r,  T[i, -1,:] * uT.to_value('K'), ls='-')
    ax.loglog(r,  T[i, 0,:] * uT.to_value('K'), ls='--')
ax.loglog(r, 350*r**-0.5, c='k', ls='--')
ax.set_xlabel('Radius [AU]')
ax.set_ylabel('T [K]')

fig.tight_layout()


In [ ]:
fig, ax = plt.subplots(ncols=1, dpi=150)

T = mu * (load3D(i=snap, var='Vc-PRS')/load3D(i=snap, var='Vc-RHO'))
T_d0 = load3D(i=snap, var='Dust0_Vc-TR0')
T_d1 = load3D(i=snap, var='Dust1_Vc-TR0')

for i in [10, 30, 60]:
    l, = ax.semilogy(z[:,i],  T[0, :,i] * uT.to_value('K'), ls='-')
    ax.semilogy(z[:,i],  T_d0[0, :,i] * uT.to_value('K'), ls='--', c=l.get_color())
    ax.semilogy(z[:,i],  T_d1[0, :,i] * uT.to_value('K'), ls=':', c=l.get_color())

ax.set_xlabel('Z [AU]')
ax.set_ylabel('T [K]')

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(ncols=1, dpi=150)

## This shows that the reason the upper layers don't equilibriate to the dust temperature is just that collisions are very slow.

for s in range(0, 11, 2):
    T = mu * (load3D(i=s, var='Vc-PRS')/load3D(i=snap, var='Vc-RHO'))
    T_d0 = load3D(i=s, var='Dust0_Vc-TR0')
    T_d1 = load3D(i=s, var='Dust1_Vc-TR0')

    i = 30
    l, = ax.semilogy(z[:,i],  T[0, :,i] * uT.to_value('K'), ls='-')
    ax.semilogy(z[:,i],  T_d0[0, :,i] * uT.to_value('K'), ls='--', c=l.get_color())
    ax.semilogy(z[:,i],  T_d1[0, :,i] * uT.to_value('K'), ls=':', c=l.get_color())

ax.set_xlabel('Z [AU]')
ax.set_ylabel('T [K]')

fig.tight_layout()